# VELNOR General v1 — Free GPU Training

This notebook runs the repository's general-purpose rsLoRA experiment on a **free CUDA GPU runtime**. It uses Qwen2.5-1.5B-Instruct, 288 training examples, 30 held-out validation examples, and a 27-case cross-domain evaluation benchmark.

## Before running
1. In Google Colab, select **Runtime → Change runtime type → T4 GPU** (or another free CUDA GPU if offered).
2. Run the cells in order. The preflight stops if CUDA is unavailable or less than 8 GiB of GPU memory is free.
3. Training uses the full dataset by default. Do not enable `VELNOR_FAST_TRAIN` for a real training run.
4. At the end, download the adapter archive or optionally copy it to Google Drive.

This is the 1.5B general-instruction experiment, not the separate Qwen3-8B QLoRA experiment. It does not prove the model is production-ready; inspect the report and compare base versus tuned results.

In [ ]:
!git clone --depth 1 --branch training/general-purpose-curriculum-and-eval https://github.com/vectolith/velnor.git /content/velnor
%cd /content/velnor

In [ ]:
!pip install -q -r training/requirements.txt

In [ ]:
import torch
assert torch.cuda.is_available(), 'No CUDA GPU detected. Select a free GPU runtime and rerun.'
free_bytes, total_bytes = torch.cuda.mem_get_info()
free_gb = free_bytes / (1024 ** 3)
total_gb = total_bytes / (1024 ** 3)
print('GPU:', torch.cuda.get_device_name(0))
print(f'Free memory: {free_gb:.2f} GiB / {total_gb:.2f} GiB')
assert free_gb >= 8, 'VELNOR requires at least 8 GiB free GPU memory for this run.'


In [ ]:
%cd /content/velnor/training
!VELNOR_BASE_MODEL=Qwen/Qwen2.5-1.5B-Instruct VELNOR_MIN_GPU_FREE_GB=8 python train_velnor_lora.py

In [ ]:
!python grade_generations.py outputs/velnor-general-v1/report.json

In [ ]:
from pathlib import Path
import shutil
run_dir = Path('outputs/velnor-general-v1')
assert (run_dir / 'report.json').exists(), 'Training report is missing.'
assert (run_dir / 'adapter' / 'adapter_config.json').exists(), 'Adapter config is missing.'
archive = shutil.make_archive('/content/velnor-general-v1', 'zip', root_dir=run_dir)
print('Verified report and adapter. Archive:', archive)
print('Report:', (run_dir / 'report.json').resolve())

In [ ]:
from google.colab import files
files.download('/content/velnor-general-v1.zip')

## Optional: persist a copy in Google Drive

Run this only if you want a Drive backup. It will request Google Drive authorisation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
destination = Path('/content/drive/MyDrive/VELNOR/general-v1')
destination.mkdir(parents=True, exist_ok=True)
shutil.copy2('/content/velnor-general-v1.zip', destination / 'velnor-general-v1.zip')
shutil.copy2(run_dir / 'report.json', destination / 'report.json')
print('Saved training archive and report to:', destination)